# Bitcoin Data and Exploratory Analysis

## Role
Canonical UTC data preparation and split definition.

## Inputs
`data/bitcoin/btcusd_1-min_data.csv`

## Outputs
Executed dataset, missingness, aggregation, return, volatility, and split summaries.

## Depends On
Raw Bitcoin source data.

## Authoritative Status
AUTHORITATIVE DATA PREPARATION

## What This Notebook Does Not Do
It does not train models or write forecast artifacts.


In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.bitcoin_pipeline import *
RUN_GENERATION = False
PROMOTE_TO_AUTHORITATIVE = False


In [2]:
daily, target = load_bitcoin_target(ROOT)
train, test = canonical_split(target)
raw = load_bitcoin_data(ROOT/'data'/'bitcoin'/'btcusd_1-min_data.csv')
summary = pd.DataFrame({'Value':[len(raw), raw.Timestamp.min(), raw.Timestamp.max(), len(daily), target.index.min(), target.index.max(), target.isna().sum(), target.index.duplicated().sum()]}, index=['Raw rows','Raw start','Raw end','Daily rows','Daily start','Daily end','Missing Close','Duplicate daily dates'])
summary

,Value
Raw rows,7633557
Raw start,2012-01-01 00:01:00+00:00
Raw end,2026-07-07 01:57:00+00:00
Daily rows,5302
Daily start,2012-01-01 00:00:00+00:00
Daily end,2026-07-07 00:00:00+00:00
Missing Close,0
Duplicate daily dates,0


## Daily aggregation
Open=first, High=max, Low=min, Close=last, Volume=sum in UTC calendar days. The target is the last available Close in each UTC date.

In [3]:
pd.DataFrame({'Start':[train.index.min(),test.index.min()],'End':[train.index.max(),test.index.max()],'Rows':[len(train),len(test)]}, index=['Train','Test'])

,Start,End,Rows
Train,2012-01-01 00:00:00+00:00,2023-08-11 00:00:00+00:00,4241
Test,2023-08-12 00:00:00+00:00,2026-07-07 00:00:00+00:00,1061


In [4]:
eda = pd.DataFrame({'Close':target,'Log Return':np.log(target/target.shift(1))}); eda['30-Day Volatility']=eda['Log Return'].rolling(30).std(); eda.tail()

,Close,Log Return,30-Day Volatility
Timestamp,,,
2026-07-03 00:00:00+00:00,62522.46,0.016829,0.019967
2026-07-04 00:00:00+00:00,63086.18,0.008976,0.020038
2026-07-05 00:00:00+00:00,63587.06,0.007908,0.018272
2026-07-06 00:00:00+00:00,64000.10,0.006475,0.018276
2026-07-07 00:00:00+00:00,63902.77,-0.001522,0.016819


## Limitation
The final UTC date contains only the available observations through 01:57 UTC and therefore represents a partial daily observation rather than a completed 24-hour UTC trading day.